## #01. 준비작업 

### 1. 공간 분석을 위한 패키지 설치 

In [11]:
# "!"로 시작되는 경우 터미널 명령으로 인식함 
# "-q" 옵션은 설치과정 출력을 생략하라는 의미 (quiet)
!pip install --upgrade -q geopandas libpysal pyproj esda


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. 라이브러리 참조 

In [12]:
from jussam import load_data
from helpers import *
from pandas import concat, DataFrame
from IPython.display import display, Markdown

# 공간특성의 존재 확인을 위한 라이브러리 참조 
from geopandas import GeoDataFrame, read_file, points_from_xy
from libpysal.weights import Queen, Rook, KNN
from esda.moran import Moran
from pyproj import CRS

### 3. 품질 점검 완료 데이터셋 불러오기 

In [13]:
origin = load_data("california_housing_qtcheck")
num_desc = load_data("california_housing_numerical_summary")
cat_desc = load_data("california_housing_categorical_summary")

📚 캘리포니아 주택 가격 데이터셋의 품질 검사 완료 버전 (출처: 자체 정제)

    field               description
--  ------------------  --------------
 0  longitude           경도
 1  latitude            위도
 2  housing_median_age  주택 중위 연령
 3  total_rooms         총 방 수
 4  total_bedrooms      총 침실 수
 5  population          인구
 6  households          가구 수
 7  median_income       중위 소득
 8  median_house_value  주택 중위 가격
 9  ocean_proximity     해양 근처 위치

📚 캘리포니아 주택 가격 데이터셋의 연속형 변수에 대한 기술 통계량 (출처: 자체 작업)
📚 캘리포니아 주택 가격 데이터셋의 범주형 변수에 대한 기술 통계량 (출처: 자체 작업)


### 4. 범주형 타입 변환 및 서열 척도 지정 

In [14]:
df = my_qtcheck.set_type(origin, as_category=cat_desc.columns)

# 서열척도를 적용할 변수와 순서를 정의한다 (낮은등급 -> 높은등급)
# -> 도메인 지식이 필요한 부분이므로 #01(준비작업)에서만 수정한다 
# -> 서열척도 대상이 없는 데이터셋이라면 제외하면 된다 
ordinal_map = {}

# 정의된 내용에 따라 순서를 재지정한다 
# -> 데이터셋에 없는 컬럼은 건너뛰므로 다른 데이터셋에서도 오류가 나지 않는다 
for col, order in ordinal_map.items():
    if col in df.columns:
        df[col] = df[col].cat.reorder_categories(order, ordered=True)

<class 'pandas.DataFrame'>
RangeIndex: 20433 entries, 0 to 20432
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           20433 non-null  float64 
 1   latitude            20433 non-null  float64 
 2   housing_median_age  20433 non-null  int64   
 3   total_rooms         20433 non-null  int64   
 4   total_bedrooms      20433 non-null  float64 
 5   population          20433 non-null  int64   
 6   households          20433 non-null  int64   
 7   median_income       20433 non-null  float64 
 8   median_house_value  20433 non-null  int64   
 9   ocean_proximity     20433 non-null  category
dtypes: category(1), float64(4), int64(5)
memory usage: 1.4 MB


### 5. 변수 유형 분류 

In [15]:
target = "median_house_value" # 종속변수 이름
target_is_continuous = True # 종속변수 연속형 여부 

# 분석 대상에서 제외할 변수
# -> 식별자(ID), 다른 변수와 중복되는 항목, 일반 변수로 다룰 수 없는 값 등을 나열한다 
# -> 이번 데이터셋에서는 위경도가 여기에 해당한다 
exclude_cols = ["longitude", "latitude"]    

nominal_cols = my_qtcheck.get_categorical_column_names(df) # 범주형 변수
continuous_cols = my_qtcheck.get_number_column_names(df) # 연속형 변수 

if target_is_continuous: # 종속변수가 연속형 변수라면? 
    continuous_cols.remove(target) # 연속형 변수 이름에서 종속변수 항목 제거 
else:                              # 종속변수가 범주형 변수라면?
    nominal_cols.remove(target) # 범주형 변수 이름에서 종속변수 항목 제거 

# 제외 대상으로 지정한 변수를 각 목록에서 제거한다 
# -> 범주형에도 적용하므로 제외대상이 범주형인 경우에도 그대로 동작한다.
continuous_cols = [c for c in continuous_cols if c not in exclude_cols] 
nominal_cols = [c for c in nominal_cols if c not in exclude_cols]

print("종속변수: ", target)
print("종속변수 유형: " + ("연속형" if target_is_continuous else "범주형"))    
print("범주형: ", nominal_cols)
print("연속형: ", continuous_cols)
print("제외 : ", exclude_cols)

종속변수:  median_house_value
종속변수 유형: 연속형
범주형:  ['ocean_proximity']
연속형:  ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
제외 :  ['longitude', 'latitude']


### 6. 컬럼 의미를 정리한 딕셔너리 

In [16]:
column_means = {
    "median_house_value": "주택 중위 가격 (USD)",
    "longitude":           "경도",
    "latitude":            "위도",
    "housing_median_age":  "주택 중위 연령",
    "total_rooms":         "총 방 수",
    "total_bedrooms":      "총 침실 수",
    "population":          "인구",
    "households":          "가구 수",
    "median_income":       "중위 소득",
    "ocean_proximity":     "해양 근접도"
}